In [0]:
import smtplib
from email.mime.text import MIMEText
from pyspark.sql.functions import col

## Fetching data from PostgreSQL(acting like Greenplum)(legacy Data Warehouse)

In [0]:
jdbcUrl = "jdbc:postgresql://ecommerce-server.postgres.database.azure.com:5432/ecommerce_greenplum_db"
jdbc_username = dbutils.secrets.get(scope="postgre_scope", key="user-name")
jdbc_password = dbutils.secrets.get(scope="postgre_scope", key="password")
jdbc_driver = "org.postgresql.Driver"

df_legacy = spark.read.format("jdbc")\
                .option("url", jdbcUrl)\
                .option("user", jdbc_username)\
                .option("password", jdbc_password)\
                .option("driver", jdbc_driver)\
                .option("dbtable", "legacy_greenplum_orders")\
                .load()


In [0]:
display(df_legacy)

## Fetching data from snowflake(New DataWarehouse)

In [0]:
sfUser = dbutils.secrets.get(scope="migrationSnowflakeScope", key="username")
sfPassword = dbutils.secrets.get(scope="migrationSnowflakeScope", key="password")

connection_options = {
    "sfUrl" : "DAPPSDF-RV26711.snowflakecomputing.com",
    "sfUser" : sfUser,
    "sfPassword" : sfPassword,
    "sfDatabase" : "ECOMMERCE_DB",
    "sfSchema" : "MIGRATION",
    "sfWarehouse" : "COMPUTE_WH"
}

df_new = spark.read.format("snowflake")\
                    .options(**connection_options)\
                    .option('dbtable', "TARGET_ORDERS")\
                    .load()


In [0]:
display(df_new)

## Function for Alert Email
If the validation for reconciliation fails

In [0]:
def send_error_email(gp_count, sf_count, join_count):
    """Sends an alert email to the ETL team using Mailtrap Sandbox."""
    sender = "data-alerts@yourcompany.com"
    recipients = ["etl-team@yourcompany.com"]

    msg_body = f"""
    Halt! Data Validation Failed during Greenplum to Snowflake Reconciliation.
    
    Reconciliation Metrics:
    - Legacy Row Count (Greenplum): {gp_count}
    - New Row Count (Snowflake): {sf_count}
    - Successfully Matched Rows (Inner Join): {join_count}
    - Mismatch Delta: {gp_count - join_count} records are broken.
    
    Discrepancy Detected: There are uncommon or missing records between systems. 
    Please investigate the upstream ETL pipelines immediately.
    """

    msg = MIMEText(msg_body)
    msg["Subject"] = "CRITICAL: ETL Data Reconciliation Failure"
    msg["From"] = sender
    msg["To"] = ", ".join(recipients)

    try:
        # Your specific Mailtrap Sandbox details
        smtp_server = "sandbox.smtp.mailtrap.io"
        port = 2525
        
        # We put your exact username here
        mailtrap_user = "03f415c03c6e69" 
        
        # REPLACE THIS with the fully revealed password from your screen!
        mailtrap_password = "c8fc7a2b523a78" 

        print("\n[SYSTEM] Connecting to Mailtrap to send alert...")
        
        with smtplib.SMTP(smtp_server, port) as server:
            server.login(mailtrap_user, mailtrap_password)
            server.sendmail(sender, recipients, msg.as_string())
            
        print("[SYSTEM] Alert email successfully caught by Mailtrap!")
        
    except Exception as e:
        print(f"[SYSTEM ERROR] Failed to send email alert: {str(e)}")

## Reconciliation Logic

In [0]:
# Getting count of rows for both Data Warehouse tables
legacy_count = df_legacy.count()
new_count = df_new.count()

print(f"Legacy Row Count (Greenplum): {legacy_count}")
print(f"New Row Count (Snowflake): {new_count}")

df_joined = df_legacy.join(df_new, ['md5'], "inner")

# Getting count of joined_df, that is how many common records do they have
join_count = df_joined.count()

print(f"Successfully Matched Rows (Inner Join): {join_count}")

# VALIDATION CHECK & ALERT TRIGGER
if legacy_count == new_count == join_count:
    print("\nSTATUS: SUCCESS! Data reconciliation passed perfectly.")
else:
    print("\nSTATUS: CRITICAL WARNING! Data mismatch detected.")
    send_error_email(legacy_count, new_count, join_count)